# LegalRAG-TR — Demo & Sonuçlar Notebook'u

**Türkçe Yargıtay & Danıştay Kararları için Domain-Adaptive Embedding + Hybrid Retrieval RAG Sistemi**
**Veri:** [`erdem-erdem/Turkish-Law-Documents-700k-clustered`](https://huggingface.co/datasets/erdem-erdem/Turkish-Law-Documents-700k-clustered) (HF, açık)

**Akademik katkılar:**
1. `multilingual-e5-small` modelinin Türkçe hukuk üzerinde domain-adaptive fine-tuning'i (MNRL)
2. 15 sorguluk manuel doğrulanmış mini benchmark + standart retrieval metrikleri
3. Hibrit retrieval (BM25 + dense + RRF) + opsiyonel cross-encoder reranker
4. Halüsinasyon-kontrollü extractive cevap üretimi

## 1. Ortam

In [ ]:
import os, sys
os.environ['PYTHONNOUSERSITE'] = '1'
sys.path.insert(0, 'src')

import torch
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}')

## 2. Açık Kaynak Veri Seti (HuggingFace)

In [ ]:
from hf_loader import load_hf_decisions

sample = load_hf_decisions(n=5)
for r in sample[:2]:
    print('---')
    print(f"Kurul: {r['kurul']}")
    print(f"Esas/Karar: {r['esas_no']} / {r['karar_no']} ({r['tarih']})")
    print(f"Konu: {r['konu']}")
    print(f"Metin uzunluğu: {len(r['text'])} karakter")
    print(f"Excerpt: {r['text'][:250]}...")

## 3. Yüklü İndeksler

Daha önce build edilmiş 3 farklı indeks:

| İndeks | Model | Boyut |
|---|---|---|
| `index_v1` | e5-base baseline | 768d |
| `index_small_base` | e5-small baseline | 384d |
| `index_small_ft` | e5-small + FT (önerilen) | 384d |

In [ ]:
from embedder import load_model, load_index
from retriever import HybridRetriever

# Önerilen: fine-tuned model + onun indeksi
INDEX_DIR = 'artifacts/index_small_ft'
MODEL_NAME = 'artifacts/e5-small-tr-legal-ft'

index, embeddings, records = load_index(INDEX_DIR)
model = load_model(MODEL_NAME)
retriever = HybridRetriever(index, embeddings, records, model)

n_kararlar = len(set(r['karar_id'] for r in records))
print(f'İndeks: {n_kararlar} karar, {len(records)} chunk, {embeddings.shape[1]}d')
print(f'Model: {MODEL_NAME}')

## 4. Canlı Sorgu Testleri (Hibrit Retrieval)

In [ ]:
from answer import format_answer
from IPython.display import Markdown, display

def ask(q, top_k=5):
    res = retriever.search(q, top_k=top_k)
    display(Markdown(format_answer(q, res, max_results=top_k)))

ask('İhtiyati haciz icra takibi sayılır mı?')

In [ ]:
ask('Karşılıksız yararlanma suçunda beraat hangi koşullarda mümkündür?')

In [ ]:
ask('2004 sayılı İcra İflas Kanunu 264. madde')   # leksik test (BM25 katkısı)

In [ ]:
ask('İdari yargıda yürütmenin durdurulması koşulları nelerdir?')

## 5. Sistem Karşılaştırması (Benchmark Sonuçları)

5 farklı konfigürasyon, 15 manuel doğrulanmış sorgu üzerinde.

In [ ]:
import json
import pandas as pd

with open('artifacts/benchmark_results.json', encoding='utf-8') as f:
    results = json.load(f)

rows = []
for name, data in results.items():
    agg = data['agg']
    rows.append({
        'System': name,
        'Recall@1': round(agg['recall@1'], 3),
        'Recall@3': round(agg['recall@3'], 3),
        'Recall@5': round(agg['recall@5'], 3),
        'MRR': round(agg['mrr'], 3),
        'nDCG@5': round(agg['ndcg@5'], 3),
        'nDCG@10': round(agg['ndcg@10'], 3),
        'Latency (ms)': round(agg['avg_latency_ms'], 0),
    })
df = pd.DataFrame(rows)
df

## 6. Fine-Tuning Etkisi (Akademik Ana Bulgu)

**e5-small baseline → e5-small + FT** karşılaştırması.

| Metrik | Baseline | Fine-Tuned | Δ |
|---|---:|---:|---:|
| Recall@1 | 0.733 | **0.933** | **+27 %** |
| MRR | 0.833 | **0.933** | +12 % |
| nDCG@5 | 0.850 | **0.913** | +7 % |
| nDCG@10 | 0.836 | **0.887** | +6 % |
| Latency | 22 ms | 15 ms | -32 % |

**Sonuç:** Türkçe hukuk üzerinde domain-adaptive fine-tuning ilk doğru kararı %27 daha sık üst sıraya getiriyor. Üstelik daha düşük gecikme.

**Önemli bonus bulgu:** Fine-tuned e5-small (30M parametre, Recall@1=0.933) generic e5-base'i (110M parametre, Recall@1=0.800) **geçti**. Yani domain adaptation, model boyutunu telafi etti.

## 7. BM25 vs Dense vs Hibrit (Bileşen Analizi)

In [ ]:
q = '2004 sayılı İcra İflas Kanunu 264. madde'
dense_only = retriever._dense_search(q, 5)
sparse_only = retriever._bm25_search(q, 5)
hybrid = retriever.search(q, top_k=5)

print('=== DENSE ONLY (anlamsal) ===')
for i, (idx, score) in enumerate(dense_only, 1):
    r = records[idx]
    print(f'  [{i}] {r["kurul"]} E.{r["esas_no"]} | score={score:.3f}')

print('\n=== BM25 ONLY (leksikal) ===')
for i, (idx, score) in enumerate(sparse_only, 1):
    r = records[idx]
    print(f'  [{i}] {r["kurul"]} E.{r["esas_no"]} | bm25={score:.2f}')

print('\n=== HYBRID (BM25 + Dense + RRF) ===')
for i, r in enumerate(hybrid, 1):
    print(f'  [{i}] {r["kurul"]} E.{r["esas_no"]} | rrf={r["score"]:.4f}')

## 8. Sonraki Adımlar

1. **Tam 700k karara ölçeklenme** (FAISS HNSW/IVF-PQ)
2. **Türkçe için cross-encoder reranker fine-tuning** (mevcut reranker yardımcı olmadı — net işaret)
3. **Açık Türkçe hukuk retrieval benchmarkı** (200-500 elle annotated)
4. **e5-base + LoRA fine-tuning** (6GB GPU'ya sığacak)
5. **Türkçe LLM ile faithfulness-controlled generative özetleme**